# ATU: Automated Trading Unit - Master Google Colab GPU Workflow

This notebook integrates the **ATU** Reinforcement Learning trading system with Google Colab GPU compute, GitHub source synchronization, and Google Drive persistent storage.

### Pipeline Architecture:
- **Development Workstation:** Google Antigravity IDE (Windows)
- **Version Control & Synchronization:** GitHub (`https://github.com/SUZUB/ATU.git`)
- **Compute Runtime:** Google Colab GPU (NVIDIA T4 / V100 / A100)
- **Persistent Storage:** Google Drive Workspace (`/content/drive/MyDrive/ATU_Workspace`)
  - **Datasets:** `/content/drive/MyDrive/ATU_Workspace/datasets`
  - **Checkpoints:** `/content/drive/MyDrive/ATU_Workspace/checkpoints`
  - **Results & Logs:** `/content/drive/MyDrive/ATU_Workspace/results`

### Workflow Outline:
1. Environment Initialization
2. Python Version Verification
3. GPU Detection
4. NVIDIA GPU Information
5. PyTorch CUDA Verification
6. GitHub Repository Synchronization (Public Clone & Pull)
7. Google Drive Authentication
8. Persistent Drive Workspace Configuration
9. Dependency Installation
10. Dataset Validation (`DATASET_PATH`)
11. Project Initialization & Module Imports
12. FinBERT Sentiment Initialization
13. Market Data Processing (5-min OHLCV)
14. Feature Engineering (Technical Indicators + Sentiment)
15. PPO Environment Initialization (Gymnasium Rules Engine)
16. Training Phase: 1,000-Step Initial Smoke Test
17. Full Training Execution (200,000 Timesteps with Checkpoints)
18. Evaluation & Out-of-Sample Continuous Backtest
19. Institutional Financial Metrics & Results CSV Persistence
20. Google Drive Workspace Verification
21. Equity Curve Visualization & Artifact Packaging

## 1. Environment Initialization
Prepare the working directories and runtime environment.

In [ ]:
import os
import sys
from pathlib import Path

WORK_DIR = Path("/content/ATU")
print(f"Workspace Root: {WORK_DIR}")
print(f"System Path:    {sys.path[0]}")

## 2. Python Version Verification
Confirm Python version compatibility (>= 3.10 required).

In [ ]:
import platform

py_version = sys.version.split()[0]
print(f"Python Version: {py_version}")
print(f"Platform:       {platform.platform()}")

assert sys.version_info >= (3, 10), f"Python >= 3.10 required, detected: {py_version}"
print("Python version verification PASSED.")

## 3. GPU Detection
Check for active CUDA GPU runtime in Google Colab.

In [ ]:
import torch

cuda_is_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_is_available}")

if not cuda_is_available:
    print("WARNING: CUDA is not active! Switch to GPU runtime via Runtime -> Change runtime type -> T4 GPU.")
else:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    print(f"GPU Devices:    {device_count}")
    print(f"Active Device:  [{current_device}] {device_name}")

## 4. NVIDIA GPU Information
Query GPU hardware specifications, VRAM, and driver details.

In [ ]:
if torch.cuda.is_available():
    dev_idx = torch.cuda.current_device()
    dev_name = torch.cuda.get_device_name(dev_idx)
    vram_bytes = torch.cuda.get_device_properties(dev_idx).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"GPU Name:     {dev_name}")
    print(f"Total VRAM:   {vram_gb:.2f} GB")
    print(f"Capability:   {torch.cuda.get_device_capability(dev_idx)}")
    !nvidia-smi
else:
    print("No CUDA GPU detected. nvidia-smi unavailable.")

## 5. PyTorch CUDA Verification
Verify tensor allocation and CUDA kernel execution.

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version:    {torch.version.cuda}")

if torch.cuda.is_available():
    t = torch.randn((1000, 1000), device="cuda")
    t_prod = torch.matmul(t, t)
    print(f"PyTorch CUDA tensor benchmark passed. Device: {t_prod.device}")
else:
    t = torch.randn((100, 100))
    print("PyTorch CPU tensor test passed.")

## 6. GitHub Repository Synchronization
Standard public clone and pull mechanism for `https://github.com/SUZUB/ATU.git`.

In [ ]:
import os
if not os.path.exists('ATU'):
    !git clone https://github.com/SUZUB/ATU.git
%cd ATU
!git pull origin main

## 7. Google Drive Authentication
Mount Google Drive for persistent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 8. Persistent Drive Workspace Configuration
Define working directories for datasets, checkpoints, logs, and results in Google Drive.

In [ ]:
import os
# Configure this to the exact folder name created in the user's Google Drive
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Compatibility pointers for Path-based references
from pathlib import Path
DRIVE_DATA_DIR = Path(DATASET_PATH)
DRIVE_MODELS_DIR = Path(MODEL_CHECKPOINTS)

print(f"Drive Workspace:    {DRIVE_WORKSPACE}")
print(f"Datasets Path:      {DATASET_PATH}")
print(f"Checkpoints Path:   {MODEL_CHECKPOINTS}")
print(f"Results Path:       {RESULTS_PATH}")

## 9. Dependency Installation
Install pinned production dependencies from `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

## 10. Dataset Validation (Google Drive Workspace)
Inspect and validate the historical financial news datasets in `DATASET_PATH` (`/content/drive/MyDrive/ATU_Workspace/datasets`).

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Search in DATASET_PATH, or fallback to legacy ATU_Data / local archive
candidate_paths = [
    Path(DATASET_PATH) / "analyst_ratings_processed.csv",
    Path("/content/drive/MyDrive/ATU_Data/analyst_ratings_processed.csv"),
    Path("/content/drive/MyDrive/ATU_Workspace/analyst_ratings_processed.csv"),
    Path("archive/analyst_ratings_processed.csv"),
]
dataset_path = next((p for p in candidate_paths if p.exists() and p.stat().st_size > 0), candidate_paths[0])
print(f"Target Dataset Path: {dataset_path}")

# 1. Verify existence & non-zero size
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset file missing: {dataset_path}\n"
        f"Please upload 'analyst_ratings_processed.csv' to your Google Drive folder:\n"
        f"  '{DATASET_PATH}'\n"
        f"Folder Link ID: 1GVxRqZmfRWnythmsGxeiIRyLMmGxxPYB"
    )

file_size_bytes = dataset_path.stat().st_size
file_size_mb = file_size_bytes / (1024 * 1024)
print(f"File Size: {file_size_mb:.2f} MB ({file_size_bytes:,} bytes)")
if file_size_bytes == 0:
    raise ValueError(f"Dataset file exists but has 0 bytes: {dataset_path}")

# 2. Inspect sample without loading entire 150MB file
sample_df = pd.read_csv(dataset_path, nrows=20)
print(f"Sample Rows Read: {len(sample_df)}")
print(f"Dataset Columns:  {list(sample_df.columns)}")

# 3. Verify columns match data processor expectations ('title' or 'headline', 'date', 'stock')
headline_col = "title" if "title" in sample_df.columns else ("headline" if "headline" in sample_df.columns else None)
date_col = "date" if "date" in sample_df.columns else None
stock_col = next((c for c in sample_df.columns if c.lower() in ["stock", "ticker"]), None)

if not headline_col:
    raise KeyError(f"Expected headline column ('title' or 'headline') not found in {list(sample_df.columns)}")
if not date_col:
    raise KeyError(f"Expected date column ('date') not found in {list(sample_df.columns)}")
if not stock_col:
    raise KeyError(f"Expected stock column ('stock' or 'ticker') not found in {list(sample_df.columns)}")

print(f"Column Mapping: headline='{headline_col}', date='{date_col}', stock='{stock_col}'")

# 4. Verify date interpretation & timezone handling
parsed_dates = pd.to_datetime(sample_df[date_col].dropna(), utc=True, errors="coerce")
if parsed_dates.isnull().all():
    raise ValueError(f"Failed to parse dates in sample column '{date_col}'")
print(f"Date Parsing: Verified (Sample range: {parsed_dates.min().strftime('%Y-%m-%d')} to {parsed_dates.max().strftime('%Y-%m-%d')})")

# 5. Check target ticker (AAPL) using chunked streaming
print("Scanning for 'AAPL' records using memory-efficient chunked reader...")
aapl_matches = []
for chunk in pd.read_csv(dataset_path, chunksize=25000):
    matches = chunk[chunk[stock_col].astype(str).str.upper() == "AAPL"]
    if not matches.empty:
        aapl_matches.append(matches)
    if len(aapl_matches) >= 5 or sum(len(m) for m in aapl_matches) >= 500:
        break

if aapl_matches:
    aapl_df = pd.concat(aapl_matches, ignore_index=True)
    aapl_dates = pd.to_datetime(aapl_df[date_col], utc=True, errors="coerce").dt.strftime("%Y-%m-%d")
    print(f"AAPL Records Found: {len(aapl_df):,} headlines in scanned chunks")
    print(f"AAPL Date Span:     {aapl_dates.min()} to {aapl_dates.max()}")
    print("Sample AAPL Headline:")
    print(f"  \"{aapl_df[headline_col].iloc[0]}\"")
else:
    raise ValueError("No records found for ticker 'AAPL' in scanned dataset chunks.")

print("\nDataset validation PASSED: Real financial news dataset is verified and ready for FinBERT!")

## 11. Project Initialization & Module Imports
Import and verify all 4 ATU core components.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from market_data_processor import MarketDataProcessor
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from train_and_backtest import create_final_state_vector
from src.data_processing import load_archive_headlines, generate_intraday_5min_market_data
from src.train import calculate_financial_metrics

print("Successfully imported all core ATU modules!")

## 12. FinBERT Sentiment Initialization
Load `ProsusAI/finbert` on GPU and score sample financial headlines.

In [ ]:
import time
import torch

target_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing SentimentProcessor on: {target_device.upper()}...")

start_t = time.time()
sp = SentimentProcessor(device=target_device)
init_time = time.time() - start_t
print(f"FinBERT loaded in {init_time:.2f} seconds. Device: {sp.device}")

# Test sample headline scoring
sample_headlines = [
    "Apple reports record quarterly revenue driven by strong iPhone sales",
    "Supply chain bottlenecks may impact tech sector margins this quarter",
    "Federal Reserve maintains interest rate benchmark steady"
]
sample_scores = sp.score_headlines(sample_headlines)
for h, s in zip(sample_headlines, sample_scores):
    print(f"  [{s:+.4f}] {h[:60]}...")

## 13. Market Data Processing (5-min OHLCV)
Process or download 5-minute intraday market bars.

In [ ]:
# Load real headlines for AAPL from Google Drive workspace
headlines = load_archive_headlines(archive_dir=DATASET_PATH, ticker="AAPL", max_rows=500, allow_synthetic_fallback=False)
print(f"Ingested {len(headlines):,} real headlines for AAPL.")

# Score sentiment with FinBERT
sentiment_df = sp.process_news_dataframe(headlines, batch_size=32)
print(f"FinBERT scored {len(headlines):,} headlines into {len(sentiment_df):,} distinct trading days.")

# Synchronize market data start date with headline date anchor
market_start_date = sentiment_df["date"].min()
print(f"Synchronizing market start date to sentiment anchor: {market_start_date}")

mdp = MarketDataProcessor()
raw_candles = generate_intraday_5min_market_data(n_rows=1500, start_date=market_start_date, initial_price=150.0)
processed_market_df = mdp.process_dataframe(raw_candles)
print(f"Market DataFrame Shape: {processed_market_df.shape}")
display(processed_market_df.head(3))

## 14. Feature Engineering (Technical Indicators + Sentiment)
Merge technical indicators (RSI, MACD, ATR, BB, ADX) with daily sentiment into `final_state_vector`.

In [ ]:
# Merge market features and FinBERT daily sentiment into unified state matrix
final_state_vector = create_final_state_vector(processed_market_df, sentiment_df)
sentiment_series = final_state_vector["daily_sentiment"]
non_zero_pct = float((sentiment_series != 0.0).mean()) * 100.0

print(f"final_state_vector Shape: {final_state_vector.shape}")
print(f"Columns: {list(final_state_vector.columns)}")
print(f"Sentiment Feature Statistics:")
print(f"  - Mean:     {sentiment_series.mean():+.4f}")
print(f"  - Std Dev:  {sentiment_series.std():.4f}")
print(f"  - Range:    [{sentiment_series.min():+.4f}, {sentiment_series.max():+.4f}]")
print(f"  - Non-Zero: {non_zero_pct:.1f}% of state vectors")

assert non_zero_pct > 0.0, "CRITICAL: Sentiment features must not be all zero!"
print("Feature engineering PASSED: Real FinBERT sentiment correctly integrated into state vectors.")

## 15. PPO Environment Initialization (Gymnasium Rules Engine)
Vectorize `ConstrainedTradingEnv` and verify observation and action spaces.

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv

# 80% Train, 20% Out-of-sample Test Split
split_idx = int(len(final_state_vector) * 0.8)
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

print(f"Train Set: {train_df.shape[0]} rows ({len(set(pd.to_datetime(train_df.index).date))} days)")
print(f"Test Set:  {test_df.shape[0]} rows ({len(set(pd.to_datetime(test_df.index).date))} days)")

# Create vectorized Gymnasium environment
train_env = DummyVecEnv([lambda: ConstrainedTradingEnv(df=train_df)])
print(f"Action Space:      {train_env.action_space}")
print(f"Observation Space: {train_env.observation_space}")

## 16. Training Phase: 1,000-Step Initial Smoke Test
Run an initial fast smoke test (1,000 steps) saving all checkpoints, tensorboard logs, and results to `DRIVE_WORKSPACE`.

In [ ]:
compute_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Launching Initial Smoke Test (1,000 timesteps) on device: {compute_device}...")
print(f"  - Model Checkpoints:  {MODEL_CHECKPOINTS}")
print(f"  - Results Directory:  {RESULTS_PATH}")
print(f"  - TensorBoard Logs:   {DRIVE_WORKSPACE}/training_logs")

# Run smoke test persisting to DRIVE_WORKSPACE subdirectories
!python train_and_backtest.py \
    --timesteps 1000 \
    --ticker AAPL \
    --device {compute_device} \
    --model-save-path "{MODEL_CHECKPOINTS}" \
    --data-dir "{DATASET_PATH}" \
    --results-dir "{RESULTS_PATH}" \
    --tensorboard-log "{DRIVE_WORKSPACE}/training_logs" \
    --finbert

## 17. Full Training Execution (200,000 Timesteps with Checkpoints)
Execute complete 200,000-step training with periodic checkpoints and tensorboard logs saved directly to `DRIVE_WORKSPACE`.
> **Note:** Run this cell after reviewing the smoke test results above.

In [ ]:
# Uncomment below when ready to run the full 200,000-step training session:
# !python train_and_backtest.py \
#     --timesteps 200000 \
#     --ticker AAPL \
#     --device {compute_device} \
#     --model-save-path "{MODEL_CHECKPOINTS}" \
#     --data-dir "{DATASET_PATH}" \
#     --results-dir "{RESULTS_PATH}" \
#     --tensorboard-log "{DRIVE_WORKSPACE}/training_logs" \
#     --checkpoint-freq 25000 \
#     --finbert

## 18. Evaluation & Out-of-Sample Continuous Backtest
Evaluate the trained model across the out-of-sample test set in an unbroken continuous compounding episode.

In [ ]:
from stable_baselines3 import PPO

model_candidates = [
    Path(MODEL_CHECKPOINTS) / "final_models" / "ppo_trading_agent_v1.zip",
    Path(MODEL_CHECKPOINTS) / "ppo_trading_agent_v1.zip",
    Path(DRIVE_WORKSPACE) / "ppo_trading_agent_v1.zip",
    Path("ppo_trading_agent_v1.zip"),
]
model_path = next((p for p in model_candidates if p.exists()), None)
if model_path is None:
    raise FileNotFoundError(
        f"Trained model checkpoint not found in {MODEL_CHECKPOINTS}.\n"
        "Please execute the Initial Smoke Test or Full Training before running this backtest cell."
    )

print(f"Loading verified model checkpoint: {model_path}")
loaded_model = PPO.load(str(model_path.with_suffix("")), device=compute_device)

test_env = ConstrainedTradingEnv(df=test_df)
state, info = test_env.reset(seed=42)

portfolio_values = [test_env.initial_balance]
step_returns = []
done = False
profitable_trades = 0

while not done:
    action, _ = loaded_model.predict(state, deterministic=True)
    state, reward, terminated, truncated, step_info = test_env.step(action)
    done = terminated or truncated
    portfolio_values.append(float(step_info["balance"]))
    step_returns.append(float(step_info["step_profit"]))
    if step_info.get("trade_closed_profit", False):
        profitable_trades += 1

print(f"Backtest evaluation completed across {len(portfolio_values)} steps.")

## 19. Institutional Financial Metrics & Results CSV Persistence
Compute financial metrics and save all performance CSVs directly to `RESULTS_PATH`.

In [ ]:
import json

metrics = calculate_financial_metrics(
    portfolio_values=portfolio_values,
    step_returns=step_returns,
    total_trades=step_info.get("total_trades_count", 0),
    profitable_trades=profitable_trades,
    initial_balance=test_env.initial_balance,
    final_balance=portfolio_values[-1],
    total_fees_paid=step_info.get("total_fees_paid", 0.0),
)

# Persist backtest CSV and metrics to RESULTS_PATH
backtest_df = pd.DataFrame({
    "step": list(range(len(portfolio_values))),
    "portfolio_value": portfolio_values,
    "step_return": [0.0] + step_returns,
})
backtest_csv_path = os.path.join(RESULTS_PATH, "backtest_portfolio_history.csv")
backtest_df.to_csv(backtest_csv_path, index=False)
print(f"Saved backtest CSV to: {backtest_csv_path}")

metrics_json_path = os.path.join(RESULTS_PATH, "evaluation_metrics.json")
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved evaluation metrics to: {metrics_json_path}")

print("=" * 65)
print("           INSTITUTIONAL PERFORMANCE METRICS REPORT           ")
print("=" * 65)
print(f" Initial Balance:        ₹{metrics['initial_balance']:,.2f}")
print(f" Final Compounded Balance:₹{metrics['final_balance']:,.2f}")
print(f" Net Profit (After Fees):₹{metrics['net_profit']:+,.2f} ({metrics['net_return_pct']:+.2f}%)")
print(f" Gross Profit (Pre-Fees):₹{metrics['gross_profit']:+,.2f} ({metrics['gross_return_pct']:+.2f}%)")
print(f" Total Brokerage & STT:  ₹{metrics['total_fees_paid']:,.2f}")
print(f" Total Trades Executed:  {metrics['total_trades']} (Win Rate: {metrics['win_rate_pct']:.1f}%)")
print(f" Maximum Drawdown:       {metrics['max_drawdown_pct']:.2f}%")
print(f" Sharpe Ratio (Ann.):    {metrics['sharpe_ratio']:.4f}")
print(f" Sortino Ratio (Ann.):   {metrics['sortino_ratio']:.4f}")
print(f" Profit Factor:          {metrics['profit_factor']:.2f}")
print("=" * 65)

## 20. Google Drive Workspace Verification
Verify persistent files in Google Drive: checkpoints, final models, tensorboard logs, and results.

In [ ]:
print("Google Drive Workspace Contents:")
!ls -lR {DRIVE_WORKSPACE}

## 21. Equity Curve Visualization & Artifact Packaging
Visualize equity curve and package all persistent Drive workspace artifacts.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(portfolio_values, color="#2563EB", linewidth=2, label="ATU Compounded Balance (₹)")
plt.axhline(test_env.initial_balance, color="gray", linestyle="--", alpha=0.7, label="Initial Capital")
plt.title("ATU Intraday Out-of-Sample Equity Curve (AAPL)", fontsize=14)
plt.xlabel("Intraday 5-Min Steps", fontsize=12)
plt.ylabel("Portfolio Balance (₹)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

chart_path = os.path.join(RESULTS_PATH, "equity_curve.png")
plt.savefig(chart_path, dpi=150)
print(f"Saved equity curve plot to: {chart_path}")
plt.show()

from google.colab import files
def download_results():
    archive_file = "/content/ATU_results.zip"
    !zip -q -r {archive_file} {DRIVE_WORKSPACE}
    files.download(archive_file)

# Run download_results() to download all workspace artifacts to your local computer:
# download_results()